# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import duckdb, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

TOKEN, REPO = userdata.get("HF_TOKEN"), "FlyRank/internship-warehouse"
FACT = hf_hub_download(REPO, "fact_content_daily_performance/month=2026-03/data_0.parquet", repo_type="dataset", token=TOKEN)
DIM_CONTENT = hf_hub_download(REPO, "dim_content.parquet", repo_type="dataset", token=TOKEN)
con = duckdb.connect()

df = con.sql(f"""
WITH early AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_early,
         SUM(gsc_clicks) AS clicks_early, AVG(gsc_avg_position) AS avg_position_early
  FROM read_parquet('{FACT}')
  WHERE gsc_data_available IS TRUE AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-20'
  GROUP BY 1, 2),
late AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_late
  FROM read_parquet('{FACT}')
  WHERE gsc_data_available IS TRUE AND report_date BETWEEN DATE '2026-03-21' AND DATE '2026-03-31'
  GROUP BY 1, 2)
SELECT e.client_hash_id, e.content_hash_id, e.impressions_early,
       e.clicks_early * 1.0 / NULLIF(e.impressions_early, 0) AS ctr_early,
       e.avg_position_early,
       DATE '2026-03-20' - c.content_updated_date AS days_since_update,
       c.word_count,
       (COALESCE(l.clicks_late, 0) / 11.0) < 0.8 * (e.clicks_early / 20.0) AS label
FROM early e
JOIN read_parquet('{DIM_CONTENT}') c USING (client_hash_id, content_hash_id)
LEFT JOIN late l USING (client_hash_id, content_hash_id)
WHERE e.clicks_early >= 5 AND c.content_updated_date <= DATE '2026-03-20'
""").df()
df["label"] = df["label"].astype(int)
df["days_since_update"] = df["days_since_update"].astype(float)
print(df.shape, "| share label=1:", round(df["label"].mean(), 3))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

(2526, 8) | share label=1: 0.59


**Signal 1: days since last update (refresh flag).** Verdict: **MIXED.** Nearly all pages were updated within the last 23 days (n=2,399), where the label rate is 0.587. The 34 pages 61+ days stale show 0.765, which matches the flag's direction, but the group is small and the 24-60 day bucket (0.591) shows no trend. Directional only.

**Signal 2 (replaces CTR): early-window impressions (volume), behind the quick-win flag.** Verdict: **OPPOSITE.** The label rate falls as volume rises, from 0.662 in the lowest bucket to 0.503 in the highest (n≈505 each). The quick-win logic favours high-volume pages, but in this sample low-volume pages carried the label more often. Part of this may be noise, because low-volume pages have noisier daily clicks and cross a relative-drop threshold more easily.

**CTR vs peers (CTR-fix flag):** FALSE, flat across buckets (0.570 to 0.626). I dropped it from the rule.

**What this means for my rule:** the rule leans mainly on low volume, with a small weight on staleness (MIXED, weak). I did not copy the quick-win direction, because the data pointed the other way.

In [9]:
bins = [-1, 23, 60, 400]
df["stale_bucket"] = pd.cut(df["days_since_update"], bins, labels=["0-23 days", "24-60 days", "61+ days"])
df.groupby("stale_bucket", observed=True)["label"].agg(n="size", rate="mean").round(3)

,n,rate
stale_bucket,,
0-23 days,2399,0.587
24-60 days,93,0.591
61+ days,34,0.765


In [11]:
df["vol_bucket"] = pd.qcut(df["impressions_early"], 5, duplicates="drop")
df.groupby("vol_bucket", observed=True)["label"].agg(n="size", rate="mean").round(3)

,n,rate
vol_bucket,,
"(55.999, 1107.0]",506,0.662
"(1107.0, 1765.0]",506,0.619
"(1765.0, 2953.0]",504,0.601
"(2953.0, 6318.0]",505,0.564
"(6318.0, 75924.0]",505,0.503


In [12]:
def pct(s): return s.rank(pct=True)

volume_part = 1 - pct(df["impressions_early"])     # lower volume -> higher score (verdict: OPPOSITE to quick-win)
stale_part  = pct(df["days_since_update"])         # staler -> higher score (verdict: MIXED, small weight)

W_VOL, W_STALE = 0.8, 0.2
df["score"] = W_VOL * volume_part + W_STALE * stale_part

df["reason_code"] = np.where(W_VOL * volume_part >= W_STALE * stale_part, "LOW_VOLUME_DECLINE_RISK", "STALE_CONTENT")
df["action"] = np.where(df["reason_code"] == "LOW_VOLUME_DECLINE_RISK", "Review page performance", "Refresh content")
print(df["reason_code"].value_counts())

reason_code
LOW_VOLUME_DECLINE_RISK    2204
STALE_CONTENT               322
Name: count, dtype: int64


In [13]:
import os, json
os.makedirs("work/outputs", exist_ok=True)

# Ranked queue (stays out of git by design; regenerated each run)
ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
queue = ranked[["client_hash_id", "content_hash_id", "score", "reason_code", "action"]]
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

# Metrics receipt: the number your Week 5 model has to beat
top50 = ranked.head(50)
metrics = {
    "n_rows": int(len(df)),
    "base_rate": round(float(df["label"].mean()), 3),
    "baseline_precision_at_50": round(float(top50["label"].mean()), 3),
}
json.dump(metrics, open("work/outputs/w04_baseline_metrics.json", "w"), indent=2)
print(metrics)

# Top 10 for your review table (hash IDs are anonymized)
ranked.head(10)[
    ["content_hash_id", "impressions_early", "days_since_update",
     "ctr_early", "avg_position_early", "reason_code", "action"]
].round(4)

{'n_rows': 2526, 'base_rate': 0.59, 'baseline_precision_at_50': 0.72}


,content_hash_id,impressions_early,days_since_update,ctr_early,avg_position_early,reason_code,action
0,content_924fcfa2714a8454,56.0,130.0,0.1429,2.4407,LOW_VOLUME_DECLINE_RISK,Review page performance
1,content_bdc656fc8f037ac0,117.0,130.0,0.0940,4.7525,LOW_VOLUME_DECLINE_RISK,Review page performance
2,content_74c25bc57a970109,205.0,25.0,0.0537,9.1582,LOW_VOLUME_DECLINE_RISK,Review page performance
3,content_bc75aa329c37f9d0,377.0,24.0,0.0212,3.4928,LOW_VOLUME_DECLINE_RISK,Review page performance
4,content_902314233f1c0b94,414.0,25.0,0.0169,4.2549,LOW_VOLUME_DECLINE_RISK,Review page performance
5,content_eb3a841aa59503d4,421.0,25.0,0.0119,8.0420,LOW_VOLUME_DECLINE_RISK,Review page performance
6,content_1b80768a4d22c0af,423.0,25.0,0.0142,4.2077,LOW_VOLUME_DECLINE_RISK,Review page performance
7,content_388fa288e762427a,439.0,24.0,0.0273,2.3615,LOW_VOLUME_DECLINE_RISK,Review page performance
8,content_ac4d98b999cd7b22,462.0,24.0,0.0152,4.2504,LOW_VOLUME_DECLINE_RISK,Review page performance
9,content_66aae6a62e195e03,486.0,25.0,0.0247,14.3912,LOW_VOLUME_DECLINE_RISK,Review page performance


## 3) Top-10 review

| # | Page (hash) | Action | Why it's here | What would make it wrong |
|---|---|---|---|---|
| 1 | content_924fcfa2 | Review page performance | Only 56 impressions, 130 days since update, CTR 0.14 at position 2.4 | With so few impressions, a handful of clicks decides the label, so a "drop" may be noise |
| 2 | content_bdc656fc | Review page performance | 117 impressions, 130 days stale, position 4.8 | A small, old page that was always quiet, so there is nothing to recover |
| 3 | content_74c25bc5 | Review page performance | 205 impressions, position 9.2, CTR 0.054 | Positions near 9 swing between page one and two, so a click drop may reflect a ranking wobble, not content |
| 4 | content_bc75aa32 | Review page performance | 377 impressions at position 3.5 with CTR 0.021, low for that position | The title may already match intent, and the low CTR could come from a search feature taking clicks |
| 5 | content_90231423 | Review page performance | 414 impressions, CTR 0.017 at position 4.3 | Same as above, and the page could have been refreshed after the snapshot |
| 6 | content_eb3a841a | Review page performance | 421 impressions, position 8.0, CTR 0.012 | A seasonal dip in a query that spikes only part of the year |
| 7 | content_1b80768a | Review page performance | 423 impressions, position 4.2, CTR 0.014 | Click counts in the early window were already low, so late-window variation looks like a decline |
| 8 | content_388fa288 | Review page performance | 439 impressions at position 2.4 but CTR 0.027, low for that rank | The page may sit under a featured answer, so a title fix would not help |
| 9 | content_ac4d98b9 | Review page performance | 462 impressions, position 4.3, CTR 0.015 | A short data gap in the daily table could look like a drop |
| 10 | content_66aae6a6 | Review page performance | 486 impressions, CTR 0.025 | The same small-page noise as the rows above |

## 4) Weak picks

All ten top picks come from the volume signal, so they are the smallest pages in the sample (56 to 486 impressions). Those are the pages where a relative-drop label is noisiest, so some of these picks may reflect random swings, not real decline. Rows 1 and 2 also have very few clicks to judge. A model that combines volume with other signals, or a label with a minimum click count in both windows, may do better than this rule.

The baseline's Precision@50 of 0.72 was measured on the same rows I used to choose the signals, so it is probably optimistic. In Week 5 I will re-score it on a grouped test split.

## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.